# **Artificial Neural Networks and Deep Learning**

---

## **Lecture 3: Cross Validation and Tuning**

<img src="https://drive.google.com/uc?export=view&id=1vR6suQUSkIpPeM5AjalVQ5wLcvglw0zf" width="400"/>



### 🌐 **Google Drive Connection**

In [ ]:
from google.colab import drive
drive.mount("/gdrive")
current_dir = "/gdrive/My\\ Drive/[2026-2027]\\ AN2DL/Lecture\\ 3"
%cd $current_dir

### ⚙️ **Libraries Import**

In [ ]:
# Install the libraries that are not preinstalled on Colab
!pip install -q torchinfo optuna

# Set seed for reproducibility
SEED = 42

import os
import re
import copy
import time
import random
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import clear_output
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary
import optuna

# Trained weights are saved in this folder, one file per model
MODELS_DIR = 'models/Cross Validation and Tuning'
os.makedirs(MODELS_DIR, exist_ok=True)


def weights_path(name):
    """File where the weights of the model called `name` are saved."""
    return f"{MODELS_DIR}/{re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_').lower()}.pt"


# Seed the Python, NumPy and PyTorch random number generators
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Use the GPU when available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Shared plot style: colour-blind-safe palette, light grid
PALETTE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']
sns.set_theme(style='whitegrid', palette=PALETTE, font_scale=1.2,
              rc={'grid.color': '#e1e0d9', 'axes.edgecolor': '#c3c2b7', 'lines.linewidth': 2})

### 📚 **Evaluation Strategies**

<img src="https://drive.google.com/uc?export=view&id=1pVmMxg_-uQJCJCAgskbS-TIIMbDYHs_B" width="600"/>


### ⏳ **Data Loading**

The Yeast dataset from the homework of Lecture 2: 1,484 proteins of baker's yeast, each described by 8 scores computed from its amino-acid sequence and labelled with one of 10 sites of the cell where the protein is found.

<img src="https://upload.wikimedia.org/wikipedia/commons/4/48/Simple_diagram_of_yeast_cell_%28en%29.svg" width="500">

In [ ]:
# Download and unzip the dataset
DATASET_NAME = "yeast.zip"
DATASET_ID = "1lTQkbiz1ed7A3Z6X2JgQd2BNIDsTS3Qn"
if not os.path.exists(DATASET_NAME):
    !gdown -q {DATASET_ID} -O {DATASET_NAME}
    !unzip -q -o {DATASET_NAME}

# yeast.data has no header: protein name, 8 scores and localisation site, separated by spaces
columns = ['Sequence Name', 'mcg', 'gvh', 'alm', 'mit', 'erl', 'pox', 'vac', 'nuc', 'Localisation Site']
dataset = pd.read_csv('yeast.data', sep=r'\s+', header=None, names=columns)
print(f"Dataset shape: {dataset.shape}")

### 🔎 **Exploration and Data Analysis**

The 8 scores, as described in `yeast.names`:
- **mcg**, **gvh**: two methods to recognise a signal sequence
- **alm**: membrane-spanning region prediction
- **mit**: amino-acid content of the N-terminal region, mitochondrial or not
- **erl**: presence of the "HDEL" signal for retention in the endoplasmic reticulum lumen (binary)
- **pox**: peroxisomal targeting signal in the C-terminus
- **vac**: amino-acid content, vacuolar or extracellular
- **nuc**: nuclear localisation signals

In [ ]:
dataset.head()

In [ ]:
# Localisation sites from the largest to the smallest
site_names = {'CYT': 'cytosolic or cytoskeletal', 'NUC': 'nuclear', 'MIT': 'mitochondrial',
              'ME3': 'membrane protein, no N-terminal signal', 'ME2': 'membrane protein, uncleaved signal',
              'ME1': 'membrane protein, cleaved signal', 'EXC': 'extracellular', 'VAC': 'vacuolar',
              'POX': 'peroxisomal', 'ERL': 'endoplasmic reticulum lumen'}
counts = dataset['Localisation Site'].value_counts()
class_names = counts.index.tolist()
num_classes = len(class_names)

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.bar(class_names, counts.values, color=PALETTE[0])
for i, count in enumerate(counts.values):
    ax.text(i, count + 5, str(count), ha='center')
ax.set(title='Proteins per localisation site', ylabel='Proteins')
ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.show()

for name in class_names:
    print(f"{name}: {site_names[name]}")
print(f"Imbalance ratio: {counts.max() / counts.min():.0f} to 1")

In [ ]:
# Distribution of each score over all the proteins
features = columns[1:9]
fig, axes = plt.subplots(2, 4, figsize=(16, 6.5))
for ax, feature in zip(axes.flat, features):
    ax.hist(dataset[feature], bins=30, color=PALETTE[0])
    ax.set(title=feature, xlim=(0, 1))
    ax.grid(axis='x', visible=False)
plt.tight_layout()
plt.show()

### 🔄 **Data Preprocessing**

In [ ]:
# Features as float32 and labels as int64, numbered from the largest site to the smallest
X = dataset[features].values.astype(np.float32)
y = dataset['Localisation Site'].map({name: i for i, name in enumerate(class_names)}).values.astype(np.int64)
input_features = X.shape[1]
print(f"Features: {X.shape}, labels: {y.shape}")

Each split has its own training, validation and test sets, drawn at random with stratification. Min-max scaling is fitted on the training set of the split only: statistics of the validation or test data would leak information about them into training.

In [ ]:
def make_split(X, y, split, batch_size, test_size, seed=SEED):
    """Loaders of one split: stratified training, validation and test sets, min-max scaled on the training set."""
    # Test set first, then the validation set from the rest; the split index changes the random draw
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed + split, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=test_size, random_state=seed + split, stratify=y_train_val
    )

    # Scaling statistics from the training set only; a feature constant there keeps its scale
    low, high = X_train.min(axis=0), X_train.max(axis=0)
    span = np.where(high > low, high - low, 1.0)
    X_train, X_val, X_test = [((part - low) / span).astype(np.float32) for part in (X_train, X_val, X_test)]

    # Only the training data are shuffled; evaluation runs in large batches, which is faster
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                              batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)), batch_size=256)
    test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test)), batch_size=256)
    return train_loader, val_loader, test_loader

In [ ]:
# Cross-validation: K random splits, each holding out 20% of the data for testing and 20% of the rest for validation
K = 3
TEST_SIZE = 0.2
BATCH_SIZE = 32

# Proteins per site in the three sets of the first split: stratification keeps the proportions, down to ERL
train_loader, val_loader, test_loader = make_split(X, y, 0, BATCH_SIZE, TEST_SIZE)
sets = {'Training': train_loader, 'Validation': val_loader, 'Test': test_loader}
table = pd.DataFrame({name: np.bincount(loader.dataset.tensors[1].numpy(), minlength=num_classes)
                      for name, loader in sets.items()}, index=class_names)
table.loc['Total'] = table.sum()
table

### 🛠️ **Model Building**

In [ ]:
class FeedForwardNet(nn.Module):
    """Fully connected network: `hidden_layers` layers of `hidden_size` neurons with ReLU and optional dropout."""

    def __init__(self, in_features, hidden_layers, hidden_size, num_classes, dropout_rate=0.0):
        super().__init__()
        layers, width = [], in_features
        for _ in range(hidden_layers):
            layers += [nn.Linear(width, hidden_size), nn.ReLU()]
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            width = hidden_size

        # Output layer: one logit per class
        layers.append(nn.Linear(width, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
def make_model(hidden_layers, hidden_size, dropout_rate=0.0):
    """Build a fresh network; reseeding gives the same initial weights to every split and configuration."""
    torch.manual_seed(SEED)
    return FeedForwardNet(input_features, hidden_layers, hidden_size, num_classes, dropout_rate).to(device)


# Starting architecture: 2 hidden layers of 128 neurons
HIDDEN_LAYERS = 2
HIDDEN_SIZE = 128
summary(make_model(HIDDEN_LAYERS, HIDDEN_SIZE), input_size=(BATCH_SIZE, input_features))

### 🧮 **Hyperparameters**

In [ ]:
LEARNING_RATE = 1e-3
EPOCHS = 100

# Metric monitored on the validation set: the sites are unbalanced, so the macro F1 score
METRIC = 'f1'

# Epochs without improvement of the validation metric before early stopping
PATIENCE = 20

criterion = nn.CrossEntropyLoss()

### 🧠 **Model Training**

The training and evaluation helpers of Lecture 2 are collapsed in the next cell: `fit` stops early on the validation metric and restores the best weights; `live` redraws the curves during training.

In [ ]:
# @title Training and evaluation helpers

def score(y_true, y_pred, metric):
    """Monitored metric: 'accuracy', or 'f1' for the F1 score averaged over the classes (macro)."""
    if metric == 'accuracy':
        return accuracy_score(y_true, y_pred)
    return f1_score(y_true, y_pred, average='macro', zero_division=0)


def train_one_epoch(model, loader, criterion, optimizer, device, metric, l1_lambda=0.0, l2_lambda=0.0):
    """Run one training epoch; return mean loss (without penalties) and the metric."""
    model.train()
    total_loss, targets, preds = 0.0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        # L1 and L2 penalties on all the parameters, added to the loss being minimised
        penalty = 0.0
        if l1_lambda > 0:
            penalty += l1_lambda * sum(p.abs().sum() for p in model.parameters())
        if l2_lambda > 0:
            penalty += l2_lambda * sum(p.pow(2).sum() for p in model.parameters())

        optimizer.zero_grad()
        (loss + penalty).backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        targets.append(y.cpu())
        preds.append(logits.argmax(dim=1).cpu())
    return total_loss / len(loader.dataset), score(torch.cat(targets), torch.cat(preds), metric)


def evaluate(model, loader, criterion, device, metric):
    """Return mean loss and the metric on a loader, without updating the weights."""
    model.eval()
    total_loss, targets, preds = 0.0, [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total_loss += criterion(logits, y).item() * x.size(0)
            targets.append(y.cpu())
            preds.append(logits.argmax(dim=1).cpu())
    return total_loss / len(loader.dataset), score(torch.cat(targets), torch.cat(preds), metric)


def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs, metric='f1',
        patience=0, l1_lambda=0.0, l2_lambda=0.0, verbose=10, live=0, track_parameters=0):
    """Train for up to `epochs` epochs and return the history.

    With patience > 0, stop after `patience` epochs without improvement of the validation
    metric and restore the weights of the best epoch. Monitoring options: a log line every
    `verbose` epochs, live plots every `live` epochs, parameters recorded every
    `track_parameters` epochs (0 switches each of them off; verbose=0 also hides the summary).
    """
    history = {'train_loss': [], 'val_loss': [], f'train_{metric}': [], f'val_{metric}': []}
    if track_parameters:
        history['parameters'] = {'epochs': [], 'values': {}, 'gradients': {}}
    best_score, best_state, wait = -np.inf, None, 0
    log = []

    for epoch in range(1, epochs + 1):
        train_loss, train_score = train_one_epoch(model, train_loader, criterion, optimizer, device,
                                                  metric, l1_lambda, l2_lambda)
        val_loss, val_score = evaluate(model, val_loader, criterion, device, metric)
        for key, value in zip(['train_loss', 'val_loss', f'train_{metric}', f'val_{metric}'],
                              [train_loss, val_loss, train_score, val_score]):
            history[key].append(value)
        if track_parameters and (epoch == 1 or epoch % track_parameters == 0):
            record_parameters(model, history['parameters'], epoch)

        # Keep a copy of the best weights seen so far
        if val_score > best_score:
            best_score, wait = val_score, 0
            history['best_epoch'] = epoch
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
        stop = patience > 0 and wait >= patience

        # Which of the log line and the live plots are due at this epoch
        last = epoch == epochs or stop
        line = (f"Epoch {epoch:3d}/{epochs} | Train: loss {train_loss:.4f}, {metric} {train_score:.4f} | "
                f"Val: loss {val_loss:.4f}, {metric} {val_score:.4f}")
        log_now = verbose > 0 and (epoch == 1 or epoch % verbose == 0 or last)
        live_now = live > 0 and (epoch == 1 or epoch % live == 0 or last)
        if log_now:
            log.append(line)
            if not live:
                print(line)

        # Redraw the plots in place, with the log below ending on the current epoch
        if live_now:
            clear_output(wait=True)
            plot_history(history, max_epochs=epochs)
            print('\n'.join(log if log_now else log + [line]))
        if stop:
            if verbose:
                print(f"Early stopping at epoch {epoch}")
            break

    # With early stopping, go back to the weights of the best epoch
    if patience > 0:
        model.load_state_dict(best_state)
    if verbose:
        kept = 'weights restored' if patience > 0 else 'weights of the last epoch kept'
        print(f"Best epoch {history['best_epoch']} | val {metric} {best_score:.4f} ({kept})")
    return history


def plot_history(history, max_epochs=None):
    """Plot training and validation curves of the loss and of the metric, marking the best epoch."""
    titles = {'loss': 'Loss', 'accuracy': 'Accuracy', 'f1': 'F1 score (macro)'}
    metrics = [key.removeprefix('train_') for key in history if key.startswith('train_')]
    fig, axes = plt.subplots(1, len(metrics), figsize=(16, 4.5))
    epochs = range(1, len(history['train_loss']) + 1)

    for ax, metric in zip(axes, metrics):
        ax.plot(epochs, history[f'train_{metric}'], color=PALETTE[0], label='Training')
        ax.plot(epochs, history[f'val_{metric}'], color=PALETTE[1], label='Validation')
        if 'best_epoch' in history:
            ax.axvline(history['best_epoch'], color='#898781', linestyle='--', linewidth=1, label='Best epoch')
        ax.set(title=titles.get(metric, metric), xlabel='Epoch', xlim=(1, max_epochs or len(epochs)))
        ax.legend()

    plt.tight_layout()
    plt.show()


def record_parameters(model, record, epoch, max_values=10_000):
    """Append the output layer's weights and bias and the gradients of every layer (at most `max_values` each)."""
    layers = [module for module in model.modules() if isinstance(module, nn.Linear)]

    def sample(tensor):
        # Copy, so that later in-place weight updates do not change the stored values
        tensor = tensor.detach().flatten()
        return tensor[::max(1, tensor.numel() // max_values)].cpu().numpy().copy()

    record['epochs'].append(epoch)
    for name, param in (('Output layer weights', layers[-1].weight), ('Output layer bias', layers[-1].bias)):
        record['values'].setdefault(name, []).append(sample(param))
    for i, layer in enumerate(layers, start=1):
        grad = torch.cat([layer.weight.grad.flatten(), layer.bias.grad.flatten()])
        record['gradients'].setdefault(f'Layer {i}', []).append(sample(grad))


def plot_histograms(tensors, epochs, kind='values'):
    """TensorBoard-style histograms of each tensor over the recorded epochs, the oldest at the back.

    For gradients, exact zeros are left out and the axis covers the central 98% of the values.
    """
    ramp = ['#cde2fb', '#0d366b'] if kind == 'values' else ['#fbd6c4', '#7a2a0a']
    colours = sns.blend_palette(ramp, as_cmap=True)
    kernel = np.exp(-0.5 * np.linspace(-2, 2, 9) ** 2)
    fig, axes = plt.subplots(1, len(tensors), figsize=(4.5 * len(tensors), 5), squeeze=False)

    for ax, (name, slices) in zip(axes[0], tensors.items()):
        if kind == 'gradients':
            # Exact zeros (neurons inactive on the batch) would dwarf the rest: leave them out
            slices = [values[values != 0] for values in slices]
        pooled = np.concatenate(slices)
        n = len(slices)

        # Values: the full range; gradients: the central 98%, so that a few outliers do not squeeze the plot.
        # The range is widened by 15% on each side so the smoothed tails are not clipped
        low, high = (pooled.min(), pooled.max()) if kind == 'values' else np.percentile(pooled, [1, 99])
        margin = 0.15 * (high - low) + 1e-8
        bins = np.linspace(low - margin, high + margin, 81)
        centres = (bins[:-1] + bins[1:]) / 2

        # Histogram of each slice, smoothed with a small Gaussian into a curve
        counts = np.stack([np.convolve(np.histogram(values, bins)[0], kernel, mode='same') for values in slices])
        heights = 5 * counts / max(counts.max(), 1)

        # One ridge per slice, from the oldest (top, back) to the newest (bottom, front)
        for k, height in enumerate(heights):
            ax.fill_between(centres, n - 1 - k, n - 1 - k + height, facecolor=colours(k / max(1, n - 1)),
                            edgecolor='white', linewidth=0.6, zorder=k)

        ticks = range(0, n, max(1, round(n / 5)))   # about five evenly spaced epoch labels
        ax.set_yticks([n - 1 - k for k in ticks], [epochs[k] for k in ticks])
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position('right')
        ax.set(title=name, xlabel='Value' if kind == 'values' else 'Gradient', ylabel='Epoch',
               xlim=(bins[0], bins[-1]))
        # A few ticks in scientific notation, so that small gradients stay readable
        ax.xaxis.set_major_locator(plt.MaxNLocator(5))
        ax.ticklabel_format(axis='x', style='sci', scilimits=(-2, 3))
        ax.grid(False)

    plt.tight_layout()
    plt.show()


def plot_parameters(history):
    """Histograms of the output layer's weights and bias, then of the gradients of every layer."""
    record = history['parameters']
    plot_histograms(record['values'], record['epochs'], kind='values')
    plot_histograms(record['gradients'], record['epochs'], kind='gradients')


def predict(model, loader, device):
    """Return true and predicted labels for every sample in a loader."""
    model.eval()
    targets, preds = [], []
    with torch.no_grad():
        for x, y in loader:
            preds.append(model(x.to(device)).argmax(dim=1).cpu())
            targets.append(y)
    return torch.cat(targets).numpy(), torch.cat(preds).numpy()


def print_metrics(y_true, y_pred):
    """Print accuracy, the macro precision, recall and F1 score, and the weighted F1 score."""
    print(f"Accuracy:            {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision (macro):   {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"Recall (macro):      {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"F1 score (macro):    {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"F1 score (weighted): {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")


def plot_confusion_matrix(y_true, y_pred, class_names, top=None, normalize=False, title='Confusion matrix'):
    """Plot a confusion matrix: rows are true classes, columns predicted classes.

    With `top`, keep the `top` largest true classes and merge all the others into 'Other'
    (so the Other/Other cell also counts mix-ups among the merged classes). With `normalize`,
    each row sums to 1 and the diagonal holds the recall of each class.
    """
    if top is not None and top < len(class_names):
        largest = np.argsort(np.bincount(y_true, minlength=len(class_names)), kind='stable')[::-1][:top]
        merge = np.full(len(class_names), top)
        merge[largest] = np.arange(top)
        y_true, y_pred = merge[y_true], merge[y_pred]
        class_names = [class_names[k] for k in largest] + ['Other']
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)),
                          normalize='true' if normalize else None)
    size = 2 + min(1.1, 7 / len(class_names)) * len(class_names)
    fig, ax = plt.subplots(figsize=(size + 1, size))
    sns.heatmap(cm, annot=True, fmt='.2f' if normalize else 'd', cmap='Blues', cbar=False,
                square=True, xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set(xlabel='Predicted label', ylabel='True label', title=title)
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    ax.tick_params(axis='y', rotation=0)
    plt.tight_layout()
    plt.show()

#### **Hold-Out: One Split**

The approach of Lecture 2: train once on the first split and judge the model on its validation set.

In [ ]:
%%time
train_loader, val_loader, test_loader = make_split(X, y, 0, BATCH_SIZE, TEST_SIZE)
model = make_model(HIDDEN_LAYERS, HIDDEN_SIZE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = fit(model, train_loader, val_loader, criterion, optimizer, device, EPOCHS, metric=METRIC,
              patience=PATIENCE, verbose=10, live=5)
torch.save(model.state_dict(), weights_path('hold-out'))

In [ ]:
y_true, y_pred = predict(model, val_loader, device)
print_metrics(y_true, y_pred)
plot_confusion_matrix(y_true, y_pred, class_names, normalize=True, title='Confusion matrix (validation set)')

#### **K-Shuffle-Split Cross-Validation**

The same configuration trained on each of the K splits. Every split starts from the same initial weights, so the differences between the splits come from the data alone. `train_on_split` trains a configuration on one split and returns its best validation score; `cross_validate` repeats it on the K splits. `live` and `track_parameters` stay off, since the searches below run hundreds of trainings.

In [ ]:
def train_on_split(X, y, config, split, criterion, device, test_size, epochs, patience, metric):
    """Train `config` on one split; return the best validation score, the history and the model."""
    train_loader, val_loader, _ = make_split(X, y, split, config['batch_size'], test_size)
    model = make_model(config['hidden_layers'], config['hidden_size'], config['dropout_rate'])
    # AdamW with weight_decay=0 is plain Adam
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'],
                                  weight_decay=config['weight_decay'])
    history = fit(model, train_loader, val_loader, criterion, optimizer, device, epochs, metric=metric,
                  patience=patience, verbose=0)
    return max(history[f'val_{metric}']), history, model


def cross_validate(X, y, config, k, **train_settings):
    """Train `config` on each of the k splits; return the best validation scores, the histories and the models."""
    scores, histories, models = zip(*[train_on_split(X, y, config, split, **train_settings) for split in range(k)])
    return np.array(scores), list(histories), list(models)

In [ ]:
%%time
# Starting configuration, changed one hyperparameter at a time in the tuning below
BASE_CONFIG = {'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE, 'hidden_layers': HIDDEN_LAYERS,
               'hidden_size': HIDDEN_SIZE, 'dropout_rate': 0.0, 'weight_decay': 0.0}

# Training settings shared by every run from here on
TRAIN_SETTINGS = {'criterion': criterion, 'device': device, 'test_size': TEST_SIZE,
                  'epochs': EPOCHS, 'patience': PATIENCE, 'metric': METRIC}

base_scores, split_histories, _ = cross_validate(X, y, BASE_CONFIG, K, **TRAIN_SETTINGS)
for split, split_score in enumerate(base_scores, start=1):
    print(f"Split {split}: validation F1 score {split_score:.4f}")
print(f"Mean over {K} splits: {base_scores.mean():.4f} ± {base_scores.std():.4f}")

In [ ]:
# Validation curves of the K splits, a dot on the best epoch of each
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for split, (colour, split_history) in enumerate(zip(PALETTE, split_histories), start=1):
    epochs = range(1, len(split_history['val_loss']) + 1)
    best_epoch = split_history['best_epoch']
    axes[0].plot(epochs, split_history['val_loss'], color=colour, linewidth=1.5, label=f'Split {split}')
    axes[1].plot(epochs, split_history[f'val_{METRIC}'], color=colour, linewidth=1.5, label=f'Split {split}')
    axes[1].scatter(best_epoch, split_history[f'val_{METRIC}'][best_epoch - 1], color=colour, s=50, zorder=3)
axes[0].set(title='Validation loss', xlabel='Epoch')
axes[1].set(title='Validation F1 score (macro)', xlabel='Epoch')
axes[0].legend()
plt.tight_layout()
plt.show()

Split 1 repeats the hold-out training above. Had we drawn a different split, the same model would have been judged differently: the spread of the K scores is the uncertainty of a single hold-out estimate.

### 🧠 **Hyperparameter Tuning**

**Grid search**: every combination of the listed values is cross-validated, and the configuration with the best mean validation score wins. We first change one hyperparameter at a time from `BASE_CONFIG`, then build on the best architecture and search learning rate and batch size together. Finally, random search and Optuna explore all six hyperparameters at once, and the three strategies are compared by the budget they use.

`grid_search` cross-validates each combination and prints its mean ± standard deviation over the splits.

In [ ]:
def grid_search(X, y, base_config, grid, k, **train_settings):
    """Cross-validate every combination of the values in `grid`, the other hyperparameters taken from `base_config`."""
    results = []
    for values in product(*grid.values()):
        changes = dict(zip(grid, values))
        config = {**base_config, **changes}
        start = time.time()
        scores, _, _ = cross_validate(X, y, config, k, **train_settings)
        results.append({'changes': changes, 'config': config, 'scores': scores, 'seconds': time.time() - start})
        print(', '.join(f"{name} {value:g}" for name, value in changes.items()) +
              f" | F1 score {scores.mean():.4f} ± {scores.std():.4f}")
    return results


def best_result(results):
    """The result with the highest mean validation score."""
    return max(results, key=lambda result: result['scores'].mean())

`plot_search` shows, for each configuration, the spread of its validation scores over the splits.

In [ ]:
def plot_search(results, title):
    """Validation scores of each configuration over the splits: box, one dot per split, mean as a diamond; the best in orange."""
    means = [result['scores'].mean() for result in results]
    best = int(np.argmax(means))
    positions = range(len(results))
    fig, ax = plt.subplots(figsize=(max(8, 1.3 * len(results) + 2), 4.8))

    boxes = ax.boxplot([result['scores'] for result in results], positions=positions, widths=0.5,
                       patch_artist=True, showfliers=False, medianprops={'color': '#2b2a26'})
    for i, box in enumerate(boxes['boxes']):
        box.set(facecolor=PALETTE[1] if i == best else PALETTE[0], alpha=0.35)
    for i, result in enumerate(results):
        ax.scatter(np.full(len(result['scores']), i), result['scores'], color='#2b2a26', s=15, zorder=3)
    ax.scatter(positions, means, marker='D', s=60, color='white', edgecolor='#2b2a26', zorder=4,
               label='Mean over the splits')

    ax.set_xticks(positions, [' / '.join(f"{value:g}" for value in result['changes'].values()) for result in results])
    ax.set(title=title, xlabel=' / '.join(results[0]['changes']), ylabel='Validation F1 score (macro)')
    ax.grid(axis='x', visible=False)
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()

`plot_heatmap` shows a two-hyperparameter grid as a table of mean ± standard deviation.

In [ ]:
def plot_heatmap(results, title):
    """Mean validation score of each cell of a two-hyperparameter grid, annotated with mean ± standard deviation; the best cell framed."""
    row, col = results[0]['changes']
    row_values = sorted({result['changes'][row] for result in results})
    col_values = sorted({result['changes'][col] for result in results})
    means = np.zeros((len(row_values), len(col_values)))
    labels = np.empty(means.shape, dtype=object)
    for result in results:
        i, j = row_values.index(result['changes'][row]), col_values.index(result['changes'][col])
        means[i, j] = result['scores'].mean()
        labels[i, j] = f"{result['scores'].mean():.3f}\n± {result['scores'].std():.3f}"

    fig, ax = plt.subplots(figsize=(2 * len(col_values) + 3, 1.2 * len(row_values) + 2))
    sns.heatmap(means, annot=labels, fmt='', cmap='Blues', ax=ax,
                xticklabels=[f"{value:g}" for value in col_values],
                yticklabels=[f"{value:g}" for value in row_values],
                cbar_kws={'label': 'Mean validation F1 score (macro)'})
    best = np.unravel_index(means.argmax(), means.shape)
    ax.add_patch(plt.Rectangle((best[1], best[0]), 1, 1, fill=False, edgecolor=PALETTE[1], linewidth=3))
    ax.set(title=title, xlabel=col, ylabel=row)
    ax.tick_params(axis='y', rotation=0)
    plt.tight_layout()
    plt.show()

#### **Batch Size**

Smaller batches make more weight updates per epoch, each computed on fewer samples and so noisier.

In [ ]:
%%time
batch_results = grid_search(X, y, BASE_CONFIG, {'batch_size': [4, 16, 32, 64, 128]}, K, **TRAIN_SETTINGS)

In [ ]:
plot_search(batch_results, 'Batch size')

#### **Learning Rate**

Within a fixed budget of epochs, a small learning rate may stop short of a good solution, and a large one may overshoot it.

In [ ]:
%%time
lr_results = grid_search(X, y, BASE_CONFIG, {'learning_rate': [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]}, K, **TRAIN_SETTINGS)

In [ ]:
plot_search(lr_results, 'Learning rate')

#### **Model Complexity**

Depth and width of the network together: 9 architectures.

In [ ]:
%%time
complexity_results = grid_search(X, y, BASE_CONFIG, {'hidden_layers': [1, 2, 3], 'hidden_size': [32, 128, 512]}, K, **TRAIN_SETTINGS)

In [ ]:
plot_search(complexity_results, 'Model complexity')

#### **Regularisation**

Dropout and AdamW weight decay (Lecture 2) on top of the best architecture found above.

In [ ]:
%%time
best_architecture = best_result(complexity_results)['config']
print(f"Best architecture: {best_architecture['hidden_layers']} hidden layers of {best_architecture['hidden_size']} neurons")
regularisation_results = grid_search(X, y, best_architecture,
                                     {'dropout_rate': [0.0, 0.2, 0.5], 'weight_decay': [0.0, 0.1, 1.0]},
                                     K, **TRAIN_SETTINGS)

In [ ]:
plot_search(regularisation_results, 'Regularisation')

#### **Learning Rate and Batch Size**

The two interact: smaller batches give noisier gradients, which tolerate smaller steps, while larger batches allow larger ones. Searching them together, on top of the best architecture and regularisation, can find combinations that the one-at-a-time searches miss. The grid includes the current values (learning rate 1e-3, batch size 32), so the search can only keep or improve the configuration found so far.

In [ ]:
%%time
best_regularised = best_result(regularisation_results)['config']
joint_results = grid_search(X, y, best_regularised,
                            {'learning_rate': [3e-4, 1e-3, 3e-3, 1e-2], 'batch_size': [8, 32, 128]},
                            K, **TRAIN_SETTINGS)

In [ ]:
plot_heatmap(joint_results, 'Learning rate and batch size')

best = best_result(joint_results)
print("Best configuration of the grid search:", best['config'])
print(f"Validation F1 score: {best['scores'].mean():.4f} ± {best['scores'].std():.4f} "
      f"(base configuration: {base_scores.mean():.4f} ± {base_scores.std():.4f})")

Mind the standard deviations. When several configurations differ by much less than their spread over the splits, the ranking among them is noise, and taking the top one partly means taking the luckiest. The shape of the heatmap, which regions are good and which are not, is more reliable than its single best cell.

#### **Random Search**

Random search draws every configuration at random from ranges instead of taking it from a fixed list: learning rate and weight decay on a logarithmic scale, the other hyperparameters uniformly. With the same number of configurations, it tries many more distinct values of each hyperparameter than a grid, which pays off when only a few of them really matter (Bergstra and Bengio, 2012). The ranges span roughly the values explored by the grids above, and all six hyperparameters change at once.

In [ ]:
def sample_config(rng):
    """A random configuration: learning rate and weight decay on a log scale, the others uniformly."""
    return {'batch_size': int(rng.choice([8, 16, 32, 64, 128])),
            'learning_rate': float(10 ** rng.uniform(-4, -2)),
            'hidden_layers': int(rng.integers(1, 4)),
            'hidden_size': int(rng.choice([32, 64, 128, 256, 512])),
            'dropout_rate': float(rng.uniform(0.0, 0.5)),
            'weight_decay': float(10 ** rng.uniform(-4, 0))}


def describe(config):
    """One-line summary of a configuration."""
    return (f"batch {config['batch_size']}, lr {config['learning_rate']:.1e}, "
            f"{config['hidden_layers']}x{config['hidden_size']}, dropout {config['dropout_rate']:.2f}, "
            f"weight decay {config['weight_decay']:.1e}")


def random_search(X, y, n_trials, k, seed=SEED, **train_settings):
    """Cross-validate `n_trials` configurations drawn at random."""
    rng = np.random.default_rng(seed)
    results = []
    for trial in range(1, n_trials + 1):
        config = sample_config(rng)
        start = time.time()
        scores, _, _ = cross_validate(X, y, config, k, **train_settings)
        results.append({'config': config, 'scores': scores, 'seconds': time.time() - start})
        print(f"Trial {trial:2d} | {describe(config)} | F1 score {scores.mean():.4f} ± {scores.std():.4f}")
    return results

In [ ]:
%%time
# The same budget for random search and Optuna: 20 configurations
N_TRIALS = 20
random_results = random_search(X, y, N_TRIALS, K, **TRAIN_SETTINGS)

#### **Optuna**

[Optuna](https://optuna.org) replaces blind sampling with a model of the search. Its default sampler, TPE (Tree-structured Parzen Estimator), splits the trials so far into good and bad ones and draws the next configuration where the good ones are more likely; the first 10 trials are random, to get it started. Optuna can also **prune** a trial: after each split it compares the running mean with the median of the earlier trials at the same point, and stops the trials that fall behind, saving the remaining trainings.

The search space is written inside the objective function, one `trial.suggest_*` call per hyperparameter, with the same ranges as the random search.

In [ ]:
# Optuna's own log is silenced: the objective prints one line per trial
optuna.logging.set_verbosity(optuna.logging.WARNING)
optuna_results = []


def objective(trial):
    """Suggest a configuration, cross-validate it split by split, and stop early if it falls behind."""
    config = {'batch_size': trial.suggest_categorical('batch_size', [8, 16, 32, 64, 128]),
              'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True),
              'hidden_layers': trial.suggest_int('hidden_layers', 1, 3),
              'hidden_size': trial.suggest_categorical('hidden_size', [32, 64, 128, 256, 512]),
              'dropout_rate': trial.suggest_float('dropout_rate', 0.0, 0.5),
              'weight_decay': trial.suggest_float('weight_decay', 1e-4, 1.0, log=True)}
    start, scores = time.time(), []
    for split in range(K):
        scores.append(train_on_split(X, y, config, split, **TRAIN_SETTINGS)[0])
        # Report the running mean after each split: the pruner decides whether to go on
        trial.report(np.mean(scores), split)
        if trial.should_prune():
            break

    scores = np.array(scores)
    pruned = len(scores) < K
    optuna_results.append({'config': config, 'scores': scores, 'seconds': time.time() - start})
    print(f"Trial {trial.number + 1:2d} | {describe(config)} | F1 score {scores.mean():.4f} ± {scores.std():.4f}"
          + (f" (pruned after {len(scores)} splits)" if pruned else ""))
    if pruned:
        raise optuna.TrialPruned()
    return scores.mean()

In [ ]:
%%time
# TPE sampler, seeded for reproducibility; median pruner, idle for the first 5 trials and the first split
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))
study.optimize(objective, n_trials=N_TRIALS)
print(f"Best configuration: {describe(study.best_params)} | F1 score {study.best_value:.4f}")

### 📈 **Results Comparison**

The three strategies against their budget, counted in trainings (one per split): grid search tried 40 configurations, random search and Optuna 30 each, and Optuna can stop trials early. Left, the best mean validation score found so far, counting only the configurations evaluated on all the splits; right, the score of every configuration, each strategy sorted from its best configuration to its worst, so that the curves show how many good configurations each strategy found.

In [ ]:
# Every grid configuration in the order it was tried, then the other two strategies
strategies = {'Grid search': batch_results + lr_results + complexity_results + regularisation_results + joint_results,
              'Random search': random_results, 'Optuna': optuna_results}

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
for colour, (name, results) in zip(PALETTE, strategies.items()):
    trainings = np.cumsum([len(result['scores']) for result in results])
    means = np.array([result['scores'].mean() for result in results])
    complete = np.array([len(result['scores']) == K for result in results])

    # Best so far among the complete configurations (a pruned trial never counts as the best)
    axes[0].step(trainings, np.fmax.accumulate(np.where(complete, means, np.nan)), where='post',
                 color=colour, label=name)
    # Every configuration, from the best to the worst; pruned trials as empty circles
    ranking = np.argsort(-means)
    ranks = np.arange(1, len(results) + 1)
    axes[1].plot(ranks, means[ranking], color=colour, linewidth=1.5, label=name)
    axes[1].scatter(ranks[complete[ranking]], means[ranking][complete[ranking]], color=colour, s=28, zorder=3)
    axes[1].scatter(ranks[~complete[ranking]], means[ranking][~complete[ranking]], facecolor='white',
                    edgecolor=colour, s=28, zorder=3)

axes[0].set(title='Best configuration so far', xlabel='Trainings (one per split)',
            ylabel='Mean validation F1 score')
axes[1].set(title='Every configuration, from the best to the worst', xlabel='Rank within the strategy',
            ylabel='Mean validation F1 score')
axes[0].legend(loc='lower right')
axes[1].scatter([], [], facecolor='white', edgecolor='#898781', s=28, label='Pruned (Optuna)')
axes[1].legend(loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:
# Budget and best configuration of each strategy
winners = {name: best_result([result for result in results if len(result['scores']) == K])
           for name, results in strategies.items()}
pd.DataFrame([{'Strategy': name,
               'Configurations': len(results),
               'Trainings': sum(len(result['scores']) for result in results),
               'Minutes': round(sum(result['seconds'] for result in results) / 60, 1),
               'Best configuration': describe(winners[name]['config']),
               'Validation F1 score': f"{winners[name]['scores'].mean():.4f} ± {winners[name]['scores'].std():.4f}"}
              for name, results in strategies.items()]).set_index('Strategy')

### 🕹️ **Inference**

The winner of each strategy is trained again on every split, and its weights are saved; then each model is loaded from its file and evaluated once on the test set of its split, which played no part in any choice. The validation scores served twice, to stop the training and to pick the configuration, so they are optimistic; the test scores are the honest estimate, and the fair ground to compare the strategies.

In [ ]:
%%time
# Train each winner on every split again, and save the weights of each model
val_results = {}
for name, winner in winners.items():
    val_results[name], _, split_models = cross_validate(X, y, winner['config'], K, **TRAIN_SETTINGS)
    for split, model in enumerate(split_models):
        torch.save(model.state_dict(), weights_path(f'{name} split {split + 1}'))

In [ ]:
# Load each model from its file and test it on its own split
test_results = {}
for name, winner in winners.items():
    config, val_scores = winner['config'], val_results[name]
    test_scores, y_true_all, y_pred_all = [], [], []
    for split in range(K):
        model = make_model(config['hidden_layers'], config['hidden_size'], config['dropout_rate'])
        model.load_state_dict(torch.load(weights_path(f'{name} split {split + 1}'), map_location=device))
        test_loader = make_split(X, y, split, config['batch_size'], TEST_SIZE)[2]
        y_true, y_pred = predict(model, test_loader, device)
        test_scores.append(score(y_true, y_pred, METRIC))
        y_true_all.append(y_true)
        y_pred_all.append(y_pred)
    test_results[name] = {'val': val_scores, 'test': np.array(test_scores),
                          'y_true': np.concatenate(y_true_all), 'y_pred': np.concatenate(y_pred_all)}
    print(f"{name:14s} | validation F1 score {val_scores.mean():.4f} ± {val_scores.std():.4f} | "
          f"test F1 score {np.mean(test_scores):.4f} ± {np.std(test_scores):.4f}")

Compare the differences between the winners with their spread over the splits: when they are smaller, the ranking on the validation sets need not match the ranking on the test sets, the same lesson as the heatmap at the level of whole strategies. Compare also the budgets: grid search spends its trainings on a fixed list, repeats and all; random search covers all six hyperparameters with fewer configurations; Optuna's pruning cuts the unpromising trials short and saves trainings.

In [ ]:
# The strategy with the best validation score: its test predictions of all the splits together
# (the test sets of different splits overlap, as they are drawn at random)
best_name = max(test_results, key=lambda name: test_results[name]['val'].mean())
print(f"Best on the validation sets: {best_name}")
print_metrics(test_results[best_name]['y_true'], test_results[best_name]['y_pred'])
plot_confusion_matrix(test_results[best_name]['y_true'], test_results[best_name]['y_pred'], class_names,
                      normalize=True, title=f'Confusion matrix (test sets of all splits, {best_name})')

#### **The Final Model**

Cross-validation has trained `K` networks with the chosen configuration, one per split, each on a different part of the data. The final model is their **ensemble**: every member scales the input with the statistics of its own training set, and the ensemble averages the class probabilities of the members. No data is left to test the ensemble itself, since every protein was used to train at least one member: its expected score on new proteins is the mean of the members' test scores.

In [ ]:
class Ensemble(nn.Module):
    """Average of the class probabilities of several networks, each with the input scaling of its own training set."""

    def __init__(self, members, shifts, scales):
        super().__init__()
        self.members = nn.ModuleList(members)
        self.register_buffer('shifts', torch.as_tensor(np.stack(shifts), dtype=torch.float32))
        self.register_buffer('scales', torch.as_tensor(np.stack(scales), dtype=torch.float32))

    def forward(self, x):
        probabilities = [torch.softmax(member((x - shift) / scale), dim=1)
                         for member, shift, scale in zip(self.members, self.shifts, self.scales)]
        return torch.stack(probabilities).mean(dim=0)


def split_scaling(X, y, split, test_size, seed=SEED):
    """Minimum and span of the features over the training set of one split, as in `make_split`."""
    X_train_val, _, y_train_val, _ = train_test_split(X, y, test_size=test_size, random_state=seed + split,
                                                      stratify=y)
    X_train, _ = train_test_split(X_train_val, test_size=test_size, random_state=seed + split,
                                  stratify=y_train_val)
    low, high = X_train.min(axis=0), X_train.max(axis=0)
    return low, np.where(high > low, high - low, 1.0)


# The members: the networks of the best strategy, one per split, loaded from their files
config = winners[best_name]['config']
members, shifts, scales = [], [], []
for split in range(K):
    member = make_model(config['hidden_layers'], config['hidden_size'], config['dropout_rate'])
    member.load_state_dict(torch.load(weights_path(f'{best_name} split {split + 1}'), map_location=device))
    low, span = split_scaling(X, y, split, TEST_SIZE)
    members.append(member.eval()), shifts.append(low), scales.append(span)
final_model = Ensemble(members, shifts, scales).to(device).eval()
torch.save(final_model.state_dict(), weights_path('final ensemble'))

scores = test_results[best_name]['test']
print(f"Final model: ensemble of {K} networks ({best_name}), saved in {weights_path('final ensemble')}")
print(f"Expected F1 score on new proteins: {scores.mean():.4f} ± {scores.std():.4f} (mean over the test sets of the splits)")

#  
<img src="https://airlab.deib.polimi.it/wp-content/uploads/2019/07/airlab-logo-new_cropped.png" width="350">

##### Connect with us:
- <img src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" width="14"> **LinkedIn:**  [AIRLab Polimi](https://www.linkedin.com/company/airlab-polimi/)
- <img src="https://upload.wikimedia.org/wikipedia/commons/9/95/Instagram_logo_2022.svg" width="14"> **Instagram:** [airlab_polimi](https://www.instagram.com/airlab_polimi/)

##### Contributors:
- **Eugenio Lomurno**: eugenio.lomurno@polimi.it
- **Alberto Archetti**: alberto.archetti@polimi.it
- **Roberto Basla**: roberto.basla@polimi.it
- **Carlo Sgaravatti**: carlo.sgaravatti@polimi.it

```
   Copyright 2026 Eugenio Lomurno, Alberto Archetti, Roberto Basla, Carlo Sgaravatti

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.
```
